# 线性回归的简洁实现

在过去的几年里，出于对深度学习强烈的兴趣，
许多公司、学者和业余爱好者开发了各种成熟的开源框架。
这些框架可以自动化基于梯度的学习算法中重复性的工作。
在上一节中，我们只运用了：

（1）通过张量来进行数据存储和线性代数；

（2）通过自动微分来计算梯度。

实际上，由于数据迭代器、损失函数、优化器和神经网络层很常用，
现代深度学习库也为我们实现了这些组件。

本节将介绍如何(**通过使用深度学习框架来简洁地实现**)
的(**线性回归模型**)。

## 生成数据集

类似，我们首先[**生成数据集**]。


In [44]:
import torch
import numpy as np
from torch.utils import data #导入数据工具包
from d2l import torch as d2l #导入d2l包并命名为d2l

true_w = torch.tensor([2,-3.4])#一维权重张量（注意：标量是0维张量，形状为()）
true_b = 4.2
features,labels = d2l.synthetic_data(true_w,true_b,1000)
features,features.shape,labels.shape,true_w

(tensor([[ 1.7326, -0.4974],
         [ 0.3572, -0.3918],
         [-1.0325,  1.1384],
         ...,
         [ 2.3441,  1.9612],
         [ 0.4692, -0.9601],
         [-0.6833,  0.9396]]),
 torch.Size([1000, 2]),
 torch.Size([1000, 1]),
 tensor([ 2.0000, -3.4000]))

## 读取数据集

我们可以[**调用框架中现有的API来读取数据**]。
我们将`features`和`labels`作为API的参数传递，并通过数据迭代器指定`batch_size`。
此外，布尔值`is_train`表示是否希望数据迭代器对象在每个迭代周期内打乱数据。


In [45]:
"""
data_arrays:你的原始数据.通常是一个包含"特征"和"标签"的元组,比如 (X, y).就像是一堆试卷(特征 X)和一堆标准答案(标签 y).
batch_size:你规定的一小把是多少个?比如 10.
is_train=True:是不是在训练?如果是训练,就需要打乱顺序(洗牌);如果是考试(测试),就不需要打乱.
"""
def load_array(data_arrays,batch_size,is_train=True):

    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset,batch_size,shuffle=is_train)

# DataLoader 在干嘛?:它是一个发牌员(数据迭代器).
# 你把刚才订好的那本练习册 (dataset) 交给发牌员.
# 如果 shuffle=True,发牌员就会把这本练习册彻底撕开,疯狂洗牌(随机打乱).
# 然后,发牌员每次给你发 batch_size 张牌(比如每次给你 10 道题和对应的 10 个答案).
# 你拿着这 10 道题去更新一次模型参数,然后再找发牌员要下一批,直到整本练习册发完.


In [46]:
batch_size = 10
data_iter = load_array((features,labels),batch_size)

使用`data_iter`的方式与3.2使用`data_iter`函数的方式相同。为了验证是否正常工作，让我们读取并打印第一个小批量样本。
在这里我们使用`iter`构造Python迭代器，并使用`next`从迭代器中获取第一项。


In [47]:
next(iter(data_iter))
# iter(data_iter) 将可迭代对象转换为迭代器(iterator)
# next() 函数用于获取迭代器的下一个元素，在这里是第一个元素

[tensor([[ 0.1416,  0.2285],
         [-1.0594,  0.5701],
         [ 1.1065,  0.0750],
         [ 0.0545,  1.2531],
         [-1.3717,  0.2236],
         [-0.9837, -1.8729],
         [ 0.4014, -1.2655],
         [ 0.1417,  0.9429],
         [ 0.5630, -0.7248],
         [ 0.4719, -0.7454]]),
 tensor([[3.7093],
         [0.1504],
         [6.1435],
         [0.0721],
         [0.6812],
         [8.6072],
         [9.3059],
         [1.2875],
         [7.7839],
         [7.6913]])]

## 定义模型

当我们在3.2中实现线性回归时，
我们明确定义了模型参数变量，并编写了计算的代码，这样通过基本的线性代数运算得到输出。
但是，如果模型变得更加复杂，且当我们几乎每天都需要实现模型时，自然会想简化这个过程。
这种情况类似于为自己的博客从零开始编写网页。
做一两次是有益的，但如果每个新博客就需要工程师花一个月的时间重新开始编写网页，那并不高效。

对于标准深度学习模型，我们可以[**使用框架的预定义好的层**]。这使我们只需关注使用哪些层来构造模型，而不必关注层的实现细节。
我们首先定义一个模型变量`net`，它是一个`Sequential`类的实例。
`Sequential`类将多个层串联在一起。
当给定输入数据时，`Sequential`实例将数据传入到第一层，
然后将第一层的输出作为第二层的输入，以此类推。
在下面的例子中，我们的模型只包含一个层，因此实际上不需要`Sequential`。
但是由于以后几乎所有的模型都是多层的，在这里使用`Sequential`会让你熟悉“标准的流水线”。

回顾3.1.2中的单层网络架构，
这一单层被称为*全连接层*（fully-connected layer），
因为它的每一个输入都通过矩阵-向量乘法得到它的每个输出。

---

在PyTorch中，全连接层在`Linear`类中定义。
值得注意的是，我们将两个参数传递到`nn.Linear`中。

1. nn.Sequential（流水线容器）
你可以把它想象成工厂里的传送带或者流水线。
在构建稍大一点的神经网络时，数据通常需要经过多道工序处理（比如：线性层 -> 激活层 -> 另一个线性层）。你只需要把这些层按顺序写进 nn.Sequential 里，数据就会自动排着队，按顺序依次通过。虽然这里目前只放了一层，但用 Sequential 包装是非常标准的现代工程写法，方便你以后随时往网络里“加层数”。

2. nn.Linear(2, 1)（全连接层 / 线性层）这是流水线上的第一道（也是目前唯一一道）工序。它在底层执行的数学逻辑，正是那个最经典的线性公式：$y = Xw + b$。
- 参数 2（输入维度 in_features）： 代表每个样本输入进来的特征数量。还记得之前定义的 true_w = torch.tensor([2, -3.4]) 吗？
- 既然权重有两个，说明每个样本有两个维度的特征，所以这里必须告诉模型输入是 2。
- 参数 1（输出维度 out_features）： 代表模型经过计算后，要吐出几个结果。在这里，无论是做回归预测还是简单的二分类，我们最终只需要得到一个具体的预测数值 $\hat{y}$，所以输出维度设定为 1。



In [48]:
# nn是神经网络的缩写
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

## (**初始化模型参数**)

在使用`net`之前，我们需要初始化模型参数。
如在线性回归模型中的权重和偏置。
深度学习框架通常有预定义的方法来初始化参数。
在这里，我们指定每个权重参数应该从均值为0、标准差为0.01的正态分布中随机采样，
偏置参数将初始化为零。


In [49]:
# net[0].weight.data.normal_(0, 0.01):
#     net[0] 表示神经网络的第一层
#     .weight 访问该层的权重参数
#     .data 访问权重参数的底层张量数据
#     .normal_(0, 0.01) 使用均值为0，标准差为0.01的正态分布来初始化权重数据
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

## 定义损失函数
[**计算均方误差使用的是`MSELoss`类，也称为平方$L_2$范数**]。
默认情况下，它返回所有样本损失的平均值。


In [50]:
loss = nn.MSELoss()

## 定义优化算法
小批量随机梯度下降算法是一种优化神经网络的标准工具，
PyTorch在`optim`模块中实现了该算法的许多变种。
当我们(**实例化一个`SGD`实例**)时，我们要指定优化的参数
（可通过`net.parameters()`从我们的模型中获得）以及优化算法所需的超参数字典。
小批量随机梯度下降只需要设置`lr`值，这里设置为0.03。


In [51]:
trainer = torch.optim.SGD(net.parameters(),lr = 0.03)

## 训练

通过深度学习框架的高级API来实现我们的模型只需要相对较少的代码。
我们不必单独分配参数、不必定义我们的损失函数，也不必手动实现小批量随机梯度下降。
当我们需要更复杂的模型时，高级API的优势将大大增加。
当我们有了所有的基本组件，[**训练过程代码与我们从零开始实现时所做的非常相似**]。

回顾一下：在每个迭代周期里，我们将完整遍历一次数据集（`train_data`），
不停地从中获取一个小批量的输入和相应的标签。
对于每一个小批量，我们会进行以下步骤:

* 通过调用`net(X)`生成预测并计算损失`l`（前向传播）。
* 通过进行反向传播来计算梯度。
* 通过调用优化器来更新模型参数。

为了更好的衡量训练效果，我们计算每个迭代周期后的损失，并打印它来监控训练过程。


In [ ]:
num_epochs = 3
for epoch in range(num_epochs):#循环3次
    for X,y in data_iter:#每次取出一批数据
        l = loss(net(X),y)
        trainer.zero_grad()
        l.backward()
        trainer.step()#更新参数
    l = loss(net(features),labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000182
epoch 2, loss 0.000098
epoch 3, loss 0.000098


下面我们[**比较生成数据集的真实参数和通过有限数据训练获得的模型参数**]。
要访问参数，我们首先从`net`访问所需的层，然后读取该层的权重和偏置。
正如在从零开始实现中一样，我们估计得到的参数与生成数据的真实参数非常接近。


In [53]:
w = net[0].weight.data
print('w的估计误差:',true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差:',true_b - b)

w的估计误差: tensor([-7.0453e-04,  1.0967e-05])
b的估计误差: tensor([-0.0004])


## 小结
* 我们可以使用PyTorch的高级API更简洁地实现模型。
* 在PyTorch中，`data`模块提供了数据处理工具，`nn`模块定义了大量的神经网络层和常见损失函数。
* 我们可以通过`_`结尾的方法将参数替换，从而初始化参数。
